### Local Concordance Calculation

In [1]:

import numpy as np
import pandas as pd

# Alternatives
alternatives = ["A0", "A1", "A2", "A3", "A4"]

# Performance matrix
# Columns: C1, C2, C3, C4, C5

performance = np.array([
    [0.78, 22_000_000, 440_000,      0, 0],
    [0.88, 21_500_000, 370_000, 30_000, 3],
    [0.83, 18_000_000, 320_000, 80_000, 5],
    [0.89, 20_000_000, 340_000, 40_000, 2],
    [0.87, 24_000_000, 500_000, 50_000, 5]
])

criteria = [
    "C1 - OTIF",
    "C2 - Holding cost",
    "C3 - Obsolescence",
    "C4 - Implementation cost",
    "C5 - Implementation time"
]

# Direction: 1 = benefit, -1 = cost
direction = np.array([1, -1, -1, -1, -1])

# Indifference thresholds
q = np.array([
    0.03,
    2_000_000,
    60_000,
    20_000,
    4
])

# Preference thresholds
p = np.array([
    0.10,
    5_000_000,
    220_000,
    50_000,
    10
])

# Calculate local concordance
local_matrices = {}

for j in range(5):

    # Difference in favour of row alternative A
    # compared with column alternative B
    difference = direction[j] * (
        performance[:, j, None]
        - performance[None, :, j]
    )

    concordance = np.where(
        difference >= -q[j],
        1,
        np.where(
            difference <= -p[j],
            0,
            (difference + p[j]) / (p[j] - q[j])
        )
    )

    # An alternative is not compared with itself
    np.fill_diagonal(concordance, np.nan)

    local_matrices[criteria[j]] = pd.DataFrame(
        concordance,
        index=alternatives,
        columns=alternatives
    )

# Display all five matrices
for criterion, matrix in local_matrices.items():

    print("\n", criterion)
    print(matrix.round(4))



 C1 - OTIF
     A0      A1      A2      A3      A4
A0  NaN  0.0000  0.7143  0.0000  0.1429
A1  1.0     NaN  1.0000  1.0000  1.0000
A2  1.0  0.7143     NaN  0.5714  0.8571
A3  1.0  1.0000  1.0000     NaN  1.0000
A4  1.0  1.0000  1.0000  1.0000     NaN

 C2 - Holding cost
     A0      A1      A2      A3   A4
A0  NaN  1.0000  0.3333  1.0000  1.0
A1  1.0     NaN  0.5000  1.0000  1.0
A2  1.0  1.0000     NaN  1.0000  1.0
A3  1.0  1.0000  1.0000     NaN  1.0
A4  1.0  0.8333  0.0000  0.3333  NaN

 C3 - Obsolescence
     A0      A1     A2     A3   A4
A0  NaN  0.9375  0.625  0.750  1.0
A1  1.0     NaN  1.000  1.000  1.0
A2  1.0  1.0000    NaN  1.000  1.0
A3  1.0  1.0000  1.000    NaN  1.0
A4  1.0  0.5625  0.250  0.375  NaN

 C4 - Implementation cost
        A0   A1   A2      A3      A4
A0     NaN  1.0  1.0  1.0000  1.0000
A1  0.6667  NaN  1.0  1.0000  1.0000
A2  0.0000  0.0  NaN  0.3333  0.6667
A3  0.3333  1.0  1.0     NaN  1.0000
A4  0.0000  1.0  1.0  1.0000     NaN

 C5 - Implementation time


### Discordance, Global Concordance, Credibility Calculation

In [ ]:
import numpy as np
import pandas as pd

# Alternatives and criteria
alternatives = ["A0", "A1", "A2", "A3", "A4"]

criteria = [
    "C1 - OTIF",
    "C2 - Holding cost",
    "C3 - Obsolescence",
    "C4 - Implementation cost",
    "C5 - Implementation time"
]

# Performance data: rows = alternatives, columns = criteria
performance = np.array([
    [0.78, 22_000_000, 440_000,      0, 0],
    [0.88, 21_500_000, 370_000, 30_000, 3],
    [0.83, 18_000_000, 320_000, 80_000, 5],
    [0.89, 20_000_000, 340_000, 40_000, 2],
    [0.87, 24_000_000, 500_000, 50_000, 5]
])

# BWM weights selected in the previous step
weights = np.array([8, 1, 4, 2, 2]) / 17

# Benefit = 1; Cost = -1
direction = np.array([1, -1, -1, -1, -1])

# Thresholds
q = np.array([0.03, 2_000_000, 60_000, 20_000, 4])
p = np.array([0.10, 5_000_000, 220_000, 50_000, 10])

# Veto thresholds: None means no veto
v = [0.20, 8_000_000, 440_000, None, None]

n_alt = len(alternatives)
n_criteria = len(criteria)

# Difference in favour of row alternative A
# compared with column alternative B
difference = np.stack([
    direction[j] * (
        performance[:, j, None] - performance[None, :, j]
    )
    for j in range(n_criteria)
], axis=2)

# ----------------------------------
# 1. Calculate local concordance
# ----------------------------------

local_concordance = np.zeros(
    (n_alt, n_alt, n_criteria)
)

for j in range(n_criteria):
    d = difference[:, :, j]

    local_concordance[:, :, j] = np.where(
        d >= -q[j],
        1,
        np.where(
            d <= -p[j],
            0,
            (d + p[j]) / (p[j] - q[j])
        )
    )

# ----------------------------------
# 2. Calculate global concordance
# ----------------------------------

global_concordance = np.sum(
    local_concordance * weights[None, None, :],
    axis=2
)

np.fill_diagonal(global_concordance, np.nan)

# ----------------------------------
# 3. Calculate local discordance
# ----------------------------------

discordance = np.zeros(
    (n_alt, n_alt, n_criteria)
)

for j in range(n_criteria):

    # Skip criteria without veto thresholds
    if v[j] is None:
        continue

    d = difference[:, :, j]

    discordance[:, :, j] = np.where(
        d <= -v[j],
        1,
        np.where(
            d >= -p[j],
            0,
            (d + p[j]) / (p[j] - v[j])
        )
    )

    np.fill_diagonal(discordance[:, :, j], np.nan)

# ----------------------------------
# 4. Calculate credibility
# ----------------------------------

credibility = np.full(
    (n_alt, n_alt), np.nan
)

adjustment_factors = np.ones(
    (n_alt, n_alt, n_criteria)
)

for a in range(n_alt):
    for b in range(n_alt):

        if a == b:
            continue

        C = global_concordance[a, b]

        for j in range(n_criteria):

            # No veto criterion: adjustment = 1
            if v[j] is None:
                continue

            D = discordance[a, b, j]

            if D > C:
                adjustment_factors[a, b, j] = (
                    (1 - D) / (1 - C)
                )

        credibility[a, b] = (
            C * np.prod(adjustment_factors[a, b, :])
        )

# ----------------------------------
# 5. Display results
# ----------------------------------

for j in range(n_criteria):

    if v[j] is None:
        print(f"\n{criteria[j]}: No veto threshold")
        continue

    print(f"\nDISCORDANCE - {criteria[j]}")

    print(pd.DataFrame(
        discordance[:, :, j],
        index=alternatives,
        columns=alternatives
    ).round(4))

print("\nGLOBAL CONCORDANCE")
print(pd.DataFrame(
    global_concordance,
    index=alternatives,
    columns=alternatives
).round(4))

print("\nCREDIBILITY MATRIX")
print(pd.DataFrame(
    credibility,
    index=alternatives,
    columns=alternatives
).round(4))

print("\nCREDIBILITY EQUALS GLOBAL CONCORDANCE:",
      np.allclose(
          credibility[~np.isnan(credibility)],
          global_concordance[~np.isnan(global_concordance)]
      ))



DISCORDANCE - C1 - OTIF
     A0   A1   A2   A3   A4
A0  NaN  0.0  0.0  0.1  0.0
A1  0.0  NaN  0.0  0.0  0.0
A2  0.0  0.0  NaN  0.0  0.0
A3  0.0  0.0  0.0  NaN  0.0
A4  0.0  0.0  0.0  0.0  NaN

DISCORDANCE - C2 - Holding cost
     A0   A1      A2   A3   A4
A0  NaN  0.0  0.0000  0.0  0.0
A1  0.0  NaN  0.0000  0.0  0.0
A2  0.0  0.0     NaN  0.0  0.0
A3  0.0  0.0  0.0000  NaN  0.0
A4  0.0  0.0  0.3333  0.0  NaN

DISCORDANCE - C3 - Obsolescence
     A0   A1   A2   A3   A4
A0  NaN  0.0  0.0  0.0  0.0
A1  0.0  NaN  0.0  0.0  0.0
A2  0.0  0.0  NaN  0.0  0.0
A3  0.0  0.0  0.0  NaN  0.0
A4  0.0  0.0  0.0  0.0  NaN

C4 - Implementation cost: No veto threshold

C5 - Implementation time: No veto threshold

GLOBAL CONCORDANCE
        A0      A1      A2      A3      A4
A0     NaN  0.5147  0.7381  0.4706  0.5966
A1  0.9608     NaN  0.9706  1.0000  1.0000
A2  0.8627  0.7479     NaN  0.7199  0.8936
A3  0.9216  1.0000  1.0000     NaN  1.0000
A4  0.8627  0.8873  0.7647  0.8137     NaN

CREDIBILITY MATRIX

### ELECTRE III Ranking

In [4]:

import numpy as np
import pandas as pd

alternatives = ["A0", "A1", "A2", "A3", "A4"]

# Credibility matrix
credibility = np.array([
    [np.nan, 0.5147, 0.7381, 0.4706, 0.5966],
    [0.9608, np.nan, 0.9706, 1.0000, 1.0000],
    [0.8627, 0.7479, np.nan, 0.7199, 0.8936],
    [0.9216, 1.0000, 1.0000, np.nan, 1.0000],
    [0.8627, 0.8873, 0.7647, 0.8137, np.nan]
])

# Lambda threshold
lam = 0.70

# Convert credibility matrix into binary outranking matrix
outranking = (credibility >= lam).astype(float)
np.fill_diagonal(outranking, np.nan)

# Replace diagonal NaN with zero for counting
binary = np.nan_to_num(outranking, nan=0.0)

# Calculate outgoing and incoming counts
outgoing = binary.sum(axis=1)
incoming = binary.sum(axis=0)

# Net outranking flow
net_flow = outgoing - incoming

results = pd.DataFrame({
    "Alternative": alternatives,
    "Outgoing count": outgoing.astype(int),
    "Incoming count": incoming.astype(int),
    "Net flow": net_flow.astype(int)
})

results["Rank"] = (
    results["Net flow"]
    .rank(ascending=False, method="min")
    .astype(int)
)

print("BINARY OUTRANKING MATRIX")
print(pd.DataFrame(
    binary,
    index=alternatives,
    columns=alternatives
).astype(int))

print("\nFINAL RANKING")
print(results.sort_values("Rank").to_string(index=False))


BINARY OUTRANKING MATRIX
    A0  A1  A2  A3  A4
A0   0   0   1   0   0
A1   1   0   1   1   1
A2   1   1   0   1   1
A3   1   1   1   0   1
A4   1   1   1   1   0

FINAL RANKING
Alternative  Outgoing count  Incoming count  Net flow  Rank
         A1               4               3         1     1
         A3               4               3         1     1
         A4               4               3         1     1
         A2               4               4         0     4
         A0               1               4        -3     5
